In [1]:
import pandas as pd
import numpy as np
import json
import warnings

# Datos de precios y volumen
adj_close = pd.read_csv("../../Datos_csv/adj_close.csv", index_col=0, parse_dates=True)
volume    = pd.read_csv("../../Datos_csv/volume.csv",    index_col=0, parse_dates=True)

# ETFs validos (cumplen estacionalidad)
with open('../../Datos_csv/tickers_estacionalidad.json', 'r', encoding='utf-8') as f:
    est_data = json.load(f)

tickers_filtrados = est_data['tickers_filtrados']
etfs_eliminados   = est_data['etfs_eliminados']

print('ETFs validos:', len(tickers_filtrados))
print('ETFs eliminados:', etfs_eliminados)

ETFs validos: 58
ETFs eliminados: ['BIL', 'EWY', 'GLD', 'IAU', 'SHY']


## Agrupación semanal

Se agrupan los retornos logarítmicos diarios por semana ISO. El retorno semanal de cada ETF se obtiene como la **media** de sus retornos logarítmicos diarios de esa semana, lo que equivale al retorno logarítmico medio diario de la semana. Esta misma lógica de agrupación se usa en el preprocesado 04 del modelo ML semanal, garantizando consistencia entre ambos enfoques.

In [ ]:
# Retornos logaritmicos diarios
ret_log_diff = np.log(adj_close).diff().iloc[1:]

# Clave semana ISO  (YYYY-Wnn)
iso_cal  = ret_log_diff.index.isocalendar()
week_key = (
    iso_cal.year.astype(str)
    + '-W'
    + iso_cal.week.astype(str).str.zfill(2)
).values

week_key_series = pd.Series(week_key, index=ret_log_diff.index, name='week_key')
print('Dias de trading:', len(ret_log_diff))
print('Semanas unicas :', len(set(week_key)))

Dias de trading: 1275
Semanas unicas : 266


In [3]:
# Retorno semanal = media de retornos diarios de esa semana (solo ETFs validos)
ret_weekly_df = (
    ret_log_diff[tickers_filtrados]
    .groupby(week_key)
    .mean()
)
ret_weekly_df.index.name = 'week_key'

print('Semanas totales:', len(ret_weekly_df))
print('Rango          :', ret_weekly_df.index[0], '->', ret_weekly_df.index[-1])
ret_weekly_df.head()

Semanas totales: 266
Rango          : 2021-W01 -> 2026-W06


,AGG,BND,DBC,DIA,DVY,EEM,EFA,EWG,EWJ,EWQ,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
week_key,,,,,,,,,,,,,,,,,,,,,
2021-W01,-0.002042,-0.002281,0.010383,0.007109,0.012052,0.012701,0.008017,0.005839,0.009299,0.006168,...,0.016233,0.021808,0.015338,0.009091,0.005529,0.000674,0.001971,0.004901,0.009639,0.014278
2021-W02,0.000290,0.000183,0.000786,-0.001804,0.002915,-0.001320,-0.003544,-0.006671,-0.001652,-0.005888,...,-0.003082,0.006321,0.000129,-0.001748,-0.005175,-0.003833,0.003778,0.002107,-0.000717,-0.003575
2021-W03,0.000000,0.000086,-0.003455,0.001497,-0.002190,0.006717,0.002508,0.005514,0.001885,0.000075,...,-0.003206,-0.004009,-0.004896,-0.000902,0.010395,-0.002142,0.003281,-0.000596,0.001322,0.006401
2021-W04,0.000102,0.000114,0.001321,-0.006665,-0.006710,-0.009238,-0.007484,-0.006812,-0.006607,-0.006620,...,-0.010318,-0.013520,-0.009379,-0.008635,-0.005960,-0.003035,-0.000326,-0.002208,-0.004402,-0.009746
2021-W05,-0.000698,-0.000760,0.008759,0.007676,0.009628,0.010701,0.006309,0.007364,0.006636,0.008167,...,0.008019,0.015836,0.012972,0.009716,0.009675,0.005145,0.006319,0.004581,0.001110,0.012235


## División Train / Test

Se usan las últimas **13 semanas** como conjunto de test, lo que equivale aproximadamente a 3 meses (62 días de trading / 5 ≈ 12,4 semanas), en línea con el corte temporal del modelo diario.

In [4]:
N_SEMANAS_TEST = 13  # ~3 meses

splits_semanal = {}
for etf in tickers_filtrados:
    serie = ret_weekly_df[etf].dropna()

    if len(serie) < 60:
        continue

    train = serie.iloc[:-N_SEMANAS_TEST]
    test  = serie.iloc[-N_SEMANAS_TEST:]

    if len(test) < 5 or len(train) < 50:
        continue

    splits_semanal[etf] = {'train': train, 'test': test}

print('ETFs con datos suficientes:', len(splits_semanal))

etf_ex = list(splits_semanal.keys())[0]
print(f'\nEjemplo ({etf_ex}):')
print(f'  Train: {splits_semanal[etf_ex]["train"].index[0]} -> '
      f'{splits_semanal[etf_ex]["train"].index[-1]} | n={len(splits_semanal[etf_ex]["train"])}')
print(f'  Test : {splits_semanal[etf_ex]["test"].index[0]} -> '
      f'{splits_semanal[etf_ex]["test"].index[-1]} | n={len(splits_semanal[etf_ex]["test"])}')

ETFs con datos suficientes: 58

Ejemplo (AGG):
  Train: 2021-W01 -> 2025-W45 | n=253
  Test : 2025-W46 -> 2026-W06 | n=13


## Búsqueda de mejores parámetros ARIMA semanal

Se realiza una búsqueda en rejilla sobre (p, q) con d=0 (los retornos ya son estacionarios). Para datos semanales se usa un horizonte de Ljung-Box de 10 retardos en lugar de 20, dado el menor número de observaciones disponibles.

In [5]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning

warnings.filterwarnings('ignore', category=ValueWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='Non-invertible starting MA parameters found.*',
                         category=UserWarning)
warnings.filterwarnings('ignore', message='Non-stationary starting autoregressive parameters found.*',
                         category=UserWarning)
# El índice semanal (ej. '2021-W01') no tiene formato fecha reconocible por statsmodels
warnings.filterwarnings('ignore', message='Could not infer format.*', category=UserWarning)

p_values = range(0, 5)
q_values = range(0, 5)
d = 0

## Búsqueda individual por ETF: AIC / BIC

Antes de hacer la búsqueda global para todos los ETFs, se analizan todas las opciones  de (p, q) para algunas ETFs. Esto permite ver cómo evolucionan AIC, BIC y el test de Ljung-Box y tomar decisiones sobre el criterio de selección.

### ¿Qué son AIC y BIC?

El AIC y el BIC son criterios para comparar modelos estadísticos. No miden si el modelo es correcto, miden qué modelo consigue mejor equilibrio entre ajustar bien los datos, sin usar demasiados parámetros.

| Criterio | Fórmula |
|----------|---------|
| **AIC** | 2k − 2ln(L) | 
| **BIC** | k·ln(n) − 2ln(L) | |

k = número de parámetros que estima el modelo
  En un ARIMA(p,0,q):
  - p coeficientes AR (φ₁, φ₂, ..., φₚ)
  - q coeficientes MA (θ₁, θ₂, ..., θq)
  - 1 varianza del error (σ²)
  - Total: k = p + q + 1

L = verosimilitud máxima del modelo (likelihood)
  Es la probabilidad de que los datos observados hayan sido generados
   por el modelo con los parámetros estimados. Cuanto mejor ajusta el
   modelo a los datos, mayor es L (y por tanto mayor ln(L) y más
  negativo −2·ln(L)).


- Cuanto más negativo, mejor.
- BIC penaliza más que AIC a medida que crece el tamaño muestral. Con pocas observaciones la diferencia es pequeña, pero BIC  favorece modelos más simples.
- Orden de selección usado en este trabajo: primero AIC (buscamos el mejor ajuste predictivo), luego BIC como desempate (penaliza la complejidad extra) y finalmente el estadístico Ljung-Box (preferimos el modelo cuyos residuos son más parecidos a ruido blanco).

### Objetivo confirmar que la mejor regla es:
1. p_value > 0.05 → los residuos son ruido blanco (condición mínima para que el modelo sea válido).
2. Entre los candidatos válidos, AIC más negativo → mejor balance ajuste/complejidad.
3. BIC como segundo criterio: si dos modelos tienen AIC similar, el de menor BIC es el más parsimonioso.
4. Estadístico Ljung-Box más pequeño → residuos más cercanos a ruido blanco puro (útil para desempatar).

In [8]:
# Exploración del grid (p, q) para los primeros 4 ETFs del conjunto
N_ETFS_EXPLORAR = 4
etfs_ejemplo = list(splits_semanal.keys())[:N_ETFS_EXPLORAR]

for etf in etfs_ejemplo:
    print(f"\n{'='*65}")
    print(f"  {etf}  —  Grid ARIMA(p, 0, q) semanal")
    print(f"{'='*65}")
    train_etf = splits_semanal[etf]['train']
    grid = []

    for p in p_values:
        for q in q_values:
            try:
                res_fit = ARIMA(train_etf, order=(p, d, q), trend='n').fit()
                lb = acorr_ljungbox(res_fit.resid.dropna(), lags=[10], return_df=True)
                grid.append({
                    'p': p, 'q': q,
                    'AIC': round(res_fit.aic, 2),
                    'BIC': round(res_fit.bic, 2),
                    'p_value': round(lb['lb_pvalue'].iloc[-1], 4),
                    'Estadístico': round(lb['lb_stat'].iloc[-1], 4)
                })
            except Exception:
                continue

    grid_df = pd.DataFrame(grid)

    # Marcar candidatos válidos (p_value > 0.05 Los residuos son ruido blanco, el modelo capturo toda la dependendia y los residuos no estan autocorrelacionados y no modelo nulo)
    grid_df['Valido'] = (
        (grid_df['p_value'] > 0.05) &
        ~((grid_df['p'] == 0) & (grid_df['q'] == 0))
    ).map({True: '✓', False: ''})

    display(grid_df.sort_values(['p', 'q']).reset_index(drop=True))

    # Mejor candidato
    cand = grid_df[(grid_df['Valido'] == '✓')]
    if not cand.empty:
        best = cand.sort_values(['BIC', 'AIC']).iloc[0]
        print(f"\n  → Seleccionado: ARIMA({int(best['p'])}, 0, {int(best['q'])}) "
              f"| AIC={best['AIC']} | BIC={best['BIC']} | p_value={best['p_value']}")
    else:
        print(f"\n  → Sin candidatos con p_value > 0.05 y orden no nulo")


  AGG  —  Grid ARIMA(p, 0, q) semanal


,p,q,AIC,BIC,p_value,Estadístico,Valido
0,0,0,-2515.09,-2511.55,0.7788,6.4201,
1,0,1,-2513.21,-2506.14,0.7846,6.3552,✓
2,0,2,-2514.27,-2503.67,0.9859,2.7928,✓
3,0,3,-2513.26,-2499.13,0.9971,1.8992,✓
4,0,4,-2511.66,-2494.00,0.9988,1.5276,✓
5,1,0,-2513.26,-2506.19,0.7907,6.2863,✓
6,1,1,-2510.66,-2500.06,0.7294,6.9578,✓
7,1,2,-2513.42,-2499.29,0.9982,1.6823,✓
8,1,3,-2511.04,-2493.37,0.9953,2.1241,✓
9,1,4,-2509.04,-2487.84,1.0000,0.5718,✓



  → Seleccionado: ARIMA(1, 0, 0) | AIC=-2513.26 | BIC=-2506.19 | p_value=0.7907

  BND  —  Grid ARIMA(p, 0, q) semanal


,p,q,AIC,BIC,p_value,Estadístico,Valido
0,0,0,-2518.65,-2515.12,0.7936,6.2533,
1,0,1,-2516.78,-2509.71,0.7960,6.2253,✓
2,0,2,-2517.95,-2507.35,0.9896,2.5813,✓
3,0,3,-2516.91,-2502.77,0.9982,1.7023,✓
4,0,4,-2515.35,-2497.68,0.9995,1.2739,✓
5,1,0,-2516.83,-2509.76,0.8026,6.1486,✓
6,1,1,-2514.25,-2503.65,0.7440,6.8020,✓
7,1,2,-2517.11,-2502.97,0.9991,1.4481,✓
8,1,3,-2514.79,-2497.13,0.9976,1.8158,✓
9,1,4,-2513.06,-2491.86,1.0000,0.4783,✓



  → Seleccionado: ARIMA(1, 0, 0) | AIC=-2516.83 | BIC=-2509.76 | p_value=0.8026

  DBC  —  Grid ARIMA(p, 0, q) semanal


,p,q,AIC,BIC,p_value,Estadístico,Valido
0,0,0,-1922.56,-1919.03,0.1229,15.2602,
1,0,1,-1922.40,-1915.34,0.2966,11.8300,✓
2,0,2,-1922.37,-1911.77,0.4910,9.4392,✓
3,0,3,-1921.66,-1907.53,0.5209,9.1180,✓
4,0,4,-1921.66,-1903.99,0.7206,7.0511,✓
5,1,0,-1922.06,-1914.99,0.2636,12.3303,✓
6,1,1,-1920.58,-1909.98,0.3116,11.6159,✓
7,1,2,-1919.98,-1905.85,0.3699,10.8422,✓
8,1,3,-1917.43,-1899.76,0.2481,12.5802,✓
9,1,4,-1924.29,-1903.09,0.9979,1.7619,✓



  → Seleccionado: ARIMA(0, 0, 1) | AIC=-1922.4 | BIC=-1915.34 | p_value=0.2966

  DIA  —  Grid ARIMA(p, 0, q) semanal


,p,q,AIC,BIC,p_value,Estadístico,Valido
0,0,0,-2040.07,-2036.53,0.4865,9.4877,
1,0,1,-2040.03,-2032.96,0.6371,7.9153,✓
2,0,2,-2038.18,-2027.58,0.6774,7.5014,✓
3,0,3,-2036.44,-2022.30,0.7254,7.0009,✓
4,0,4,-2034.40,-2016.74,0.7169,7.0903,✓
5,1,0,-2039.90,-2032.83,0.6181,8.1099,✓
6,1,1,-2038.13,-2027.53,0.6609,7.6714,✓
7,1,2,-2036.27,-2022.14,0.6885,7.3869,✓
8,1,3,-2034.45,-2016.79,0.7271,6.9827,✓
9,1,4,-2032.46,-2011.26,0.7245,7.0104,✓



  → Seleccionado: ARIMA(0, 0, 1) | AIC=-2040.03 | BIC=-2032.96 | p_value=0.6371


En este estudio de modelo semanal se ha priorizado el BIC frente al AIC porque permite seleccionar modelos más sencillos y evita añadir demasiados parámetros. Aunque el AIC puede señalar modelos con mejor ajuste, estos pueden ser más complejos. Por ello, el BIC se utiliza como criterio principal, mientras que el AIC se revisa como criterio complementario para asegurar que el modelo elegido sigue ajustándose bien a los datos.

Ej AGG: Si se seleccionase el modelo únicamente por el menor AIC, el modelo elegido sería el ARMA(2,0), ya que presenta el valor más bajo de AIC entre los modelos válidos, con -2514.32. Sin embargo, este modelo incorpora más parámetros y su BIC es de -2503.72.Si se prioriza el BIC, el modelo seleccionado sería el ARMA(1,0), ya que presenta un BIC más bajo, con un valor de -2506.19. Aunque su AIC, de -2513.26, es ligeramente peor que el del modelo ARMA(2,0), la diferencia entre ambos AIC es pequeña.el ARMA(1,0) tiene menos parámetros, por lo que es un modelo más sencillo. Por este motivo, se elige el ARMA(1,0) como modelo final.

Ej DBC : En el caso del primer ETF, el modelo ARMA(1,4) presenta el menor valor de AIC, con un valor de -1924.29. Sin embargo, este modelo incorpora un mayor número de parámetros y obtiene un BIC de -1903.09. Si eligiesemos por mejor valor de BIC, sería el modelo ARMA(0,1), este  tiene menos parámetros, su AIC es	-1922.40 que no esta muy lejos del caso (1,4) y su bic cambia considerablemente a un valor más negativo	-1915.34

Nos interesa elegir un modelo más sencillo porque evita añadir parámetros innecesarios. Un modelo demasiado complejo puede ajustarse muy bien a los datos pasados, pero no necesariamente funcionar mejor con datos futuros. Por eso, cuando la diferencia de ajuste entre dos modelos es pequeña, elegimos el modelo con menos parámetros.

In [9]:
mejores_modelos_semanal = {}

for etf, data in splits_semanal.items():
    print(f'\n Seleccionando parametros ARIMA semanal para: {etf}')
    train = data['train']
    grid  = []

    for p in p_values:
        for q in q_values:
            try:
                res = ARIMA(train, order=(p, d, q), trend='n').fit()
                lb  = acorr_ljungbox(res.resid.dropna(), lags=[10], return_df=True)

                grid.append({
                    'p': p, 'q': q, 'AIC': res.aic, 'BIC': res.bic,
                    'p_value': lb['lb_pvalue'].iloc[-1],
                    'stat':    lb['lb_stat'].iloc[-1]
                })
            except:
                continue

    grid_df = pd.DataFrame(grid)

    # Candidatos: residuos ruido blanco y modelo no trivial
    cand = grid_df[
        (grid_df['p_value'] > 0.05) &
        ~((grid_df['p'] == 0) & (grid_df['q'] == 0))
    ]

    if not cand.empty:
        best = cand.sort_values(['BIC', 'AIC']).iloc[0]
        order_elegido = (int(best['p']), 0, int(best['q']))
        mejores_modelos_semanal[etf] = {
            'order': order_elegido,
            'stats': best.to_dict()
        }
        print(f' Modelo seleccionado para {etf}: ARIMA{order_elegido}\n')
    else:
        print(f' No se encontro un modelo ARIMA semanal valido para {etf} (Ljung-Box).\n')


 Seleccionando parametros ARIMA semanal para: AGG
 Modelo seleccionado para AGG: ARIMA(1, 0, 0)


 Seleccionando parametros ARIMA semanal para: BND
 Modelo seleccionado para BND: ARIMA(1, 0, 0)


 Seleccionando parametros ARIMA semanal para: DBC
 Modelo seleccionado para DBC: ARIMA(0, 0, 1)


 Seleccionando parametros ARIMA semanal para: DIA
 Modelo seleccionado para DIA: ARIMA(0, 0, 1)


 Seleccionando parametros ARIMA semanal para: DVY
 Modelo seleccionado para DVY: ARIMA(0, 0, 1)


 Seleccionando parametros ARIMA semanal para: EEM
 Modelo seleccionado para EEM: ARIMA(0, 0, 1)


 Seleccionando parametros ARIMA semanal para: EFA
 Modelo seleccionado para EFA: ARIMA(0, 0, 1)


 Seleccionando parametros ARIMA semanal para: EWG
 Modelo seleccionado para EWG: ARIMA(0, 0, 1)


 Seleccionando parametros ARIMA semanal para: EWJ
 Modelo seleccionado para EWJ: ARIMA(0, 0, 1)


 Seleccionando parametros ARIMA semanal para: EWQ
 Modelo seleccionado para EWQ: ARIMA(0, 0, 1)


 Seleccionando para

## Modelo ARIMA / BASE / CEROS

In [10]:
resultados_preds_semanal = {}

for etf, config in mejores_modelos_semanal.items():
    print(f'Calculando predicciones semanales para: {etf}')
    order   = config['order']
    y_train = splits_semanal[etf]['train']
    y_test  = splits_semanal[etf]['test']

    # ARIMA walk-forward semana a semana
    history    = y_train.copy()
    preds_arima = []
    for val_real in y_test:
        model_fit = ARIMA(history, order=order, trend='n').fit()
        preds_arima.append(model_fit.forecast(steps=1).iloc[0])
        history = pd.concat([
            history,
            pd.Series([val_real], index=[y_test.index[len(preds_arima) - 1]])
        ])

    # BASE (Naive: retorno semana anterior)
    preds_base  = [y_train.iloc[-1]] + y_test.iloc[:-1].tolist()

    # ZEROS
    preds_zeros = np.zeros(len(y_test))

    resultados_preds_semanal[etf] = pd.DataFrame({
        'real':  y_test,
        'arima': preds_arima,
        'base':  preds_base,
        'zeros': preds_zeros
    }, index=y_test.index)

Calculando predicciones semanales para: AGG
Calculando predicciones semanales para: BND
Calculando predicciones semanales para: DBC
Calculando predicciones semanales para: DIA
Calculando predicciones semanales para: DVY
Calculando predicciones semanales para: EEM
Calculando predicciones semanales para: EFA
Calculando predicciones semanales para: EWG
Calculando predicciones semanales para: EWJ
Calculando predicciones semanales para: EWQ
Calculando predicciones semanales para: EWT
Calculando predicciones semanales para: EWU
Calculando predicciones semanales para: EWZ
Calculando predicciones semanales para: FXI
Calculando predicciones semanales para: HYG
Calculando predicciones semanales para: IEF
Calculando predicciones semanales para: IEFA
Calculando predicciones semanales para: IEMG
Calculando predicciones semanales para: IJR
Calculando predicciones semanales para: INDA
Calculando predicciones semanales para: ITOT
Calculando predicciones semanales para: IVV
Calculando predicciones sema

## Comparación de Errores

In [11]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

tablas_comparativas_semanal = {}

for etf, df in resultados_preds_semanal.items():
    y_true = df['real']

    metrics = {
        'Modelo': ['ARIMA', 'BASE', 'ZEROS'],
        'MAE': [
            mean_absolute_error(y_true, df['arima']),
            mean_absolute_error(y_true, df['base']),
            mean_absolute_error(y_true, df['zeros'])
        ],
        'RMSE': [
            root_mean_squared_error(y_true, df['arima']),
            root_mean_squared_error(y_true, df['base']),
            root_mean_squared_error(y_true, df['zeros'])
        ]
    }

    res_df = pd.DataFrame(metrics).set_index('Modelo')
    tablas_comparativas_semanal[etf] = res_df

    print(f'\nResultados semanales para {etf}:')
    display(res_df)


Resultados semanales para AGG:


,MAE,RMSE
Modelo,,
ARIMA,0.000823,0.001350
BASE,0.001023,0.001511
ZEROS,0.000825,0.001351



Resultados semanales para BND:


,MAE,RMSE
Modelo,,
ARIMA,0.000831,0.001334
BASE,0.001043,0.001485
ZEROS,0.000834,0.001335



Resultados semanales para DBC:


...

,MAE,RMSE
Modelo,,
ARIMA,0.004484,0.005867
BASE,0.006789,0.008539
ZEROS,0.004433,0.005760



Resultados semanales para XLV:


,MAE,RMSE
Modelo,,
ARIMA,0.003378,0.003980
BASE,0.004707,0.005539
ZEROS,0.003415,0.003973



Resultados semanales para XLY:


,MAE,RMSE
Modelo,,
ARIMA,0.004598,0.005555
BASE,0.006839,0.009105
ZEROS,0.004644,0.005696


In [ ]:
import matplotlib.pyplot as plt

color_real  = '#95968F'
color_arima = '#EF739A'
color_base  = '#5DE7A4'

for etf, df in resultados_preds_semanal.items():

    plt.figure(figsize=(8, 4))

    plt.plot(df['real'],       label='Real',      color=color_real,  linewidth=1.5)
    plt.plot(df['arima'] * 10, label='ARIMA*10',  color=color_arima, linewidth=1.6)
    plt.plot(df['base'],       label='Base (t-1)', color=color_base,  linewidth=1.3)

    plt.axhline(0, color='black', linewidth=0.8, alpha=0.4)
    plt.title(f'{etf} – Predicciones semanales (test)', fontsize=11)
    plt.xlabel('Semana ISO', fontsize=9)
    plt.ylabel('Retorno log medio semanal', fontsize=9)
    plt.grid(alpha=0.2)
    plt.legend(fontsize=8, frameon=False)
    plt.tight_layout()
    plt.show()

### Kappa de Cohen

Se evalúa si el modelo acierta la **dirección** del movimiento semanal (sube / baja), más allá de la magnitud del error.

In [ ]:
from sklearn.metrics import cohen_kappa_score, accuracy_score

metricas_direccionales_semanal = []

for etf, df in resultados_preds_semanal.items():
    real_dir  = (df['real']  > 0).astype(int)
    arima_dir = (df['arima'] > 0).astype(int)
    base_dir  = (df['base']  > 0).astype(int)

    try:
        kappa_arima = cohen_kappa_score(real_dir, arima_dir)
    except Exception:
        kappa_arima = float('nan')

    try:
        kappa_base = cohen_kappa_score(real_dir, base_dir)
    except Exception:
        kappa_base = float('nan')

    acc_arima = accuracy_score(real_dir, arima_dir)
    acc_base  = accuracy_score(real_dir, base_dir)

    metricas_direccionales_semanal.append({
        'ETF':            etf,
        'Kappa_ARIMA':    kappa_arima,
        'Kappa_BASE':     kappa_base,
        'Accuracy_ARIMA': f'{acc_arima:.2%}',
        'Accuracy_BASE':  f'{acc_base:.2%}'
    })

df_direccional_semanal = (
    pd.DataFrame(metricas_direccionales_semanal).set_index('ETF')
)
print('Analisis de Direccion Semanal (Sube/Baja):')
display(df_direccional_semanal)

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import os

# Guardar resultados para el notebook de analisis
records_export = []
for etf, df_etf in resultados_preds_semanal.items():
    y_true = df_etf['real']
    records_export.append({
        'ETF':          etf,
        'RMSE':          root_mean_squared_error(y_true, df_etf['arima']),
        'RMSE_baseline': root_mean_squared_error(y_true, df_etf['base']),
        'RMSE_zeros':    root_mean_squared_error(y_true, df_etf['zeros']),
        'MAE':           mean_absolute_error(y_true, df_etf['arima']),
        'MAE_baseline':  mean_absolute_error(y_true, df_etf['base']),
        'MAE_zeros':     mean_absolute_error(y_true, df_etf['zeros']),
    })

nombre_csv = 'resultados_arima_semanal.csv'
out_path = f'../../Datos_csv/{nombre_csv}'
pd.DataFrame(records_export).to_csv(out_path, index=False)
print(f'Resultados guardados en: {out_path}')


In [ ]:
from sklearn.metrics import cohen_kappa_score, accuracy_score

# Guardar metricas direccionales para el notebook de analisis
rows_dir = []
for etf, df_etf in resultados_preds_semanal.items():
    real_dir  = (df_etf['real']  > 0).astype(int)
    arima_dir = (df_etf['arima'] > 0).astype(int)
    base_dir  = (df_etf['base']  > 0).astype(int)
    try:
        kappa_a = cohen_kappa_score(real_dir, arima_dir)
    except Exception:
        kappa_a = float('nan')
    try:
        kappa_b = cohen_kappa_score(real_dir, base_dir)
    except Exception:
        kappa_b = float('nan')
    rows_dir.append({
        'ETF':            etf,
        'Kappa_ARIMA':    kappa_a,
        'Kappa_BASE':     kappa_b,
        'Accuracy_ARIMA': f'{accuracy_score(real_dir, arima_dir):.2%}',
        'Accuracy_BASE':  f'{accuracy_score(real_dir, base_dir):.2%}',
    })

out_dir = '../../Datos_csv/direccional_arima_semanal.csv'
pd.DataFrame(rows_dir).to_csv(out_dir, index=False)
print(f'Direccional guardado en: {out_dir}')

## Modelo GARCH (Volatilidad)

En finanzas la volatilidad no es constante: hay periodos de calma y de turbulencia (*volatility clustering*). ARIMA modela la **media** (dirección del retorno); GARCH modela la **varianza** (magnitud de la incertidumbre).

In [ ]:
# ! pip install arch
from arch import arch_model

In [ ]:
resultados_garch_semanal = {}

for etf, config in mejores_modelos_semanal.items():
    order   = config['order']
    y_train = splits_semanal[etf]['train']
    y_test  = splits_semanal[etf]['test']

    # Residuos del ARIMA sobre train
    arima_fit   = ARIMA(y_train, order=order, trend='n').fit()
    resid_train = arima_fit.resid.dropna()

    # GARCH(1,1) sobre los residuos de train (escala x1000 para estabilidad numerica)
    am   = arch_model(resid_train * 1000, vol='Garch', p=1, q=1, dist='normal', rescale=False)
    res  = am.fit(disp='off', show_warning=False)

    # Volatilidad condicional en test (walk-forward)
    resid_full  = arima_fit.resid.dropna()
    vol_test_list = []
    history_resid = (resid_train * 1000).values.tolist()

    for val_real in y_test:
        am_tmp = arch_model(history_resid, vol='Garch', p=1, q=1, dist='normal', rescale=False)
        try:
            res_tmp = am_tmp.fit(disp='off', show_warning=False, options={'maxiter': 200})
            fc      = res_tmp.forecast(horizon=1, reindex=False)
            vol_hat = float(fc.variance.iloc[-1, 0]) ** 0.5 / 1000  # volver a escala original
        except Exception:
            vol_hat = float('nan')
        vol_test_list.append(vol_hat)

        # Añadir residuo ARIMA del periodo actual a la historia
        arima_fit2      = ARIMA(
            pd.concat([y_train, y_test.loc[:val_real.name if hasattr(val_real, 'name') else y_test.index[0]]]),
            order=order, trend='n'
        ).fit()
        new_resid = arima_fit2.resid.dropna().iloc[-1] * 1000
        history_resid.append(new_resid)

    # Volatilidad realizada = |retorno semanal| (proxy)
    vol_realizada = y_test.abs()

    resultados_garch_semanal[etf] = pd.DataFrame({
        'vol_realizada': vol_realizada.values,
        'vol_garch':     vol_test_list,
    }, index=y_test.index)

    print(f'{etf}: GARCH(1,1) ok | params: omega={res.params[omega]:.4e}, alpha={res.params[alpha[1]]:.4f}, beta={res.params[beta[1]]:.4f}')
